# Rossmann Sales — Red Neuronal con Entradas Heterogéneas
**Persona:** Oscar &nbsp;|&nbsp; **Equipo:** Oscar · Dani · Fernando

---
Arquitectura many-to-one: rama LSTM (serie temporal) + rama densa (embeddings estáticos).  
Target: `log1p(Sales)` normalizado por tienda. Tiendas objetivo: `[1, 2, 3, 4, 5, 562, 682, 733, 769]`.

## 0 · Imports y semilla

In [ ]:
import sys
import os

# Añadir raíz del proyecto al path para importar src/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

import src.preprocessing as prep
import src.evaluate as ev
import src.model as mdl

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR    = os.path.join(PROJECT_ROOT, "data")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")

print(f"TensorFlow {tf.__version__} | NumPy {np.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU'))} dispositivo(s)")

## 1 · Carga de datos

In [ ]:
train_raw, store_raw = prep.load_raw_data(DATA_DIR)

print(f"train: {train_raw.shape} | {train_raw.Date.min().date()} → {train_raw.Date.max().date()}")
print(f"store: {store_raw.shape}")
train_raw.head(3)

## 2 · Partición temporal

Las fechas de corte están definidas en `src/evaluate.py` como contrato del equipo.  
**No redefinir aquí.**

In [ ]:
print(f"Train:  hasta            {ev.SPLIT_TRAIN_END.date()}")
print(f"Val:    {ev.SPLIT_VAL_START.date()} → {ev.SPLIT_VAL_END.date()}")
print(f"Test:   {ev.SPLIT_TEST_START.date()} → 2015-07-17   ← evaluado por el profesor")
print(f"Tiendas objetivo: {ev.TARGET_STORES}")

## 3 · Preprocesado

Implementado en `src/preprocessing.py`. Ejecutar en el orden definido por el docstring del módulo.

In [ ]:
# Limpieza: elimina Open==0 y Sales==0 con Open==1
df = prep.clean_train(train_raw)
print(f"Filas tras clean_train: {len(df):,}  (eliminadas: {len(train_raw)-len(df):,})")

In [ ]:
# Merge store.csv + imputación de nulos
df = prep.merge_store_features(df, store_raw)
print(f"Columnas tras merge: {list(df.columns)}")

In [ ]:
# Label-encoding de categóricas
df, cat_encoders = prep.encode_categoricals(df)
print(f"Encoders: {list(cat_encoders.keys())}")

In [ ]:
# Features temporales y de competencia
df = prep.engineer_features(df)
print(f"Columnas totales: {df.shape[1]}")

In [ ]:
# Split temporal
df_train, df_val, df_test = prep.temporal_split(df)
print(f"Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")

In [ ]:
# Normalización por tienda — fit SOLO sobre df_train
store_stats = prep.build_store_normalizer(df_train)
df_train = prep.normalize_sales(df_train, store_stats)
df_val   = prep.normalize_sales(df_val,   store_stats)
df_test  = prep.normalize_sales(df_test,  store_stats)

# Verificar que Sales_norm tiene media ~0 y std ~1 en train
print("Sales_norm en train — media por tienda (primeras 5):")
print(df_train.groupby("Store")["Sales_norm"].mean().head())

In [ ]:
SEQ_LEN = 30

X_seq_tr, X_static_tr, y_tr = prep.build_sequences(df_train, SEQ_LEN)
X_seq_va, X_static_va, y_va = prep.build_sequences(df_val,   SEQ_LEN)
X_seq_te, X_static_te, y_te = prep.build_sequences(df_test,  SEQ_LEN)

print(f"X_seq   train: {X_seq_tr.shape}    | dtype: {X_seq_tr.dtype}")
print(f"X_static train: {X_static_tr.shape} | dtype: {X_static_tr.dtype}")
print(f"y        train: {y_tr.shape}")

## 4 · Modelo

Arquitectura definida en `src/model.py`. Rama LSTM + rama densa con embeddings.

In [ ]:
model = mdl.build_model(
    n_stores=df["Store"].nunique(),
    seq_len=SEQ_LEN,
    n_seq_features=X_seq_tr.shape[-1],
    n_static_num_features=len(prep.STATIC_NUM_COLS),
)
model.summary()

In [ ]:
callbacks = mdl.get_callbacks(
    checkpoint_path=os.path.join(OUTPUTS_DIR, "best_model.weights.h5")
)

history = mdl.train_model(
    model,
    X_train=(X_seq_tr, X_static_tr), y_train=y_tr,
    X_val=(X_seq_va, X_static_va),   y_val=y_va,
    callbacks=callbacks,
    epochs=50,
    batch_size=256,
)

In [ ]:
# Curva de aprendizaje
plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"],     label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Época")
plt.ylabel("MSE (espacio normalizado)")
plt.legend()
plt.title("Curva de entrenamiento")
plt.tight_layout()
plt.show()

## 5 · Evaluación

R² calculado con `src/evaluate.py` — misma función para los tres miembros del equipo.

In [ ]:
y_pred_norm = model.predict([X_seq_te, X_static_te], verbose=0).flatten()

In [ ]:
report = ev.evaluation_report(df_test, y_pred_norm, store_stats)
display(report)

## 6 · Log de experimentos

Anotar aquí cada experimento antes de hacer `git push`.

| # | Fecha | Cambio principal | R² val | R² test global | Notas |
|---|-------|-----------------|--------|----------------|-------|
| 1 |       | Baseline         |        |                |       |

---
### Plantilla de nota de experimento

**Experimento #N**  
- Fecha:  
- Cambios: `(p.ej. SEQ_LEN=60, lstm_units=128)`  
- R² val:  
- R² test por tienda: *(copiar tabla del report)*  
- Observaciones: